# v4 Two-Stage Model – 최고 성능 최적화 버전

## 적용된 최적화 기법

1. **SMOTE**: Rare 샘플 증강 (149 → 5,000)
2. **하이퍼파라미터 튜닝**: RandomizedSearchCV
3. **Threshold 최적화**: Rare 민감도 조정
4. **앙상블**: XGBoost + LightGBM
5. **v4 FE 확장**: 18개 파생변수 (기존 15개 + 3개)

**예상 성능:**
- Rare F1: 0.5+ (기존 0.3 대비 67%↑)
- Macro F1: 0.58+ (기존 0.53 대비 9%↑)

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("필수 라이브러리 로드 완료!")
print("SMOTE, RandomizedSearchCV, LightGBM 준비됨")

## 1. 데이터 로드

In [ ]:
# 데이터 로드
df_master = pd.read_parquet("../data/df_master_preprocessed_v1.parquet")
print("Loaded df_master:", df_master.shape)
print("\nSegment 분포:")
print(df_master['Segment'].value_counts().sort_index())

# Top150 로드
top150_final = pd.read_parquet("../features/top150_final.parquet")
print("\nLoaded top150_final:", top150_final.shape)

## 2. v4 Feature Engineering (확장판 - 18개)

기존 15개 + 추가 3개 (dormant_score, high_value_dormant, usage_drop_ratio)

In [ ]:
# 날짜 변환 헬퍼 함수
def convert_to_date(date_col):
    """YYYYMMDD 형식을 datetime으로 변환"""
    return pd.to_datetime(date_col.astype(str), format='%Y%m%d', errors='coerce')

def days_diff(date_col, reference_date='20231231'):
    """기준일과의 차이 (일수)"""
    ref = pd.to_datetime(reference_date)
    dates = convert_to_date(date_col)
    return (ref - dates).dt.days

# v4 Feature Engineering (최적화 버전)
def create_v4_features(df):
    """
    v4 파생변수 18개 생성 (확장판)
    희귀 세그먼트(0,1) 식별에 특화
    """
    df_fe = df.copy()
    today = pd.to_datetime('20231231')
    MAX_DAYS = 18250
    
    # 1. v4_last_use_gap_CA
    if '최종이용일자_CA' in df_fe.columns:
        df_fe['v4_last_use_gap_CA'] = days_diff(df_fe['최종이용일자_CA'])
    else:
        df_fe['v4_last_use_gap_CA'] = 0
    
    # 2. v4_last_use_gap_card_all
    date_cols = ['최종이용일자_일시불', '최종이용일자_신판', '최종이용일자_할부', '최종이용일자_기본']
    available_cols = [c for c in date_cols if c in df_fe.columns]
    if available_cols:
        last_use_all = pd.DataFrame({c: convert_to_date(df_fe[c]) for c in available_cols}).max(axis=1)
        df_fe['v4_last_use_gap_card_all'] = (today - last_use_all).dt.days
    else:
        df_fe['v4_last_use_gap_card_all'] = 0
    
    # 3. v4_first_to_last_gap (극단값 처리)
    df_fe['v4_first_to_last_gap'] = 0
    if 'rv최초시작후경과일' in df_fe.columns and '최종이용일자_기본' in df_fe.columns:
        days_elapsed = df_fe['rv최초시작후경과일'].clip(upper=MAX_DAYS)
        valid_mask = (days_elapsed.notna()) & (days_elapsed > 0) & (days_elapsed <= MAX_DAYS)
        
        if valid_mask.sum() > 0:
            CHUNK_SIZE = 50000
            for i in range(0, len(df_fe), CHUNK_SIZE):
                chunk_mask = valid_mask.iloc[i:i+CHUNK_SIZE]
                if chunk_mask.sum() == 0:
                    continue
                chunk_days = days_elapsed.iloc[i:i+CHUNK_SIZE][chunk_mask]
                first_dates = today - pd.to_timedelta(chunk_days, unit='D')
                last_dates = convert_to_date(df_fe.iloc[i:i+CHUNK_SIZE].loc[chunk_mask, '최종이용일자_기본'])
                gap = (last_dates - first_dates).dt.days
                df_fe.loc[chunk_mask[chunk_mask].index, 'v4_first_to_last_gap'] = gap
    
    # 기본 사용량 계산 (재사용)
    usage_cols_r12 = ['이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_CA_R12M']
    usage_r12 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_cols_r12])
    
    usage_cols_r6 = ['이용금액_일시불_R6M', '이용금액_할부_R6M', '이용금액_CA_R6M']
    usage_r6 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_cols_r6])
    
    usage_r3_cols = ['이용금액_일시불_R3M', '이용금액_할부_R3M', '이용금액_CA_R3M']
    usage_r3 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_r3_cols])
    
    limit = df_fe['카드이용한도금액'] if '카드이용한도금액' in df_fe.columns else 1
    
    # 4. v4_limit_to_usage_ratio_R12M
    df_fe['v4_limit_to_usage_ratio_R12M'] = usage_r12 / (limit + 1e-6)
    
    # 5. v4_balance_to_usage_ratio
    balance = df_fe['평잔_6M'] if '평잔_6M' in df_fe.columns else 0
    df_fe['v4_balance_to_usage_ratio'] = balance / (usage_r6 + 1e-6)
    
    # 6. v4_bill_drop_R6_to_R3
    bill_r6 = df_fe['청구금액_R6M'] if '청구금액_R6M' in df_fe.columns else 0
    bill_r3 = df_fe['청구금액_R3M'] if '청구금액_R3M' in df_fe.columns else 0
    df_fe['v4_bill_drop_R6_to_R3'] = (bill_r6 - bill_r3) / (bill_r6 + 1e-6)
    
    # 7. v4_usage_volatility_R3_R6_R12
    vol_cols = ['이용금액_일시불_R3M', '이용금액_일시불_R6M', '이용금액_일시불_R12M']
    if all(c in df_fe.columns for c in vol_cols):
        df_fe['v4_usage_volatility_R3_R6_R12'] = df_fe[vol_cols].std(axis=1)
    else:
        df_fe['v4_usage_volatility_R3_R6_R12'] = 0
    
    # 8. v4_recent_zero_usage_flag
    df_fe['v4_recent_zero_usage_flag'] = (usage_r3 == 0).astype(int)
    
    # 9. v4_long_inactive_high_limit_flag
    high_limit = limit > 5000000
    low_usage = usage_r12 < 100000
    long_inactive = df_fe['v4_last_use_gap_card_all'] > 180
    df_fe['v4_long_inactive_high_limit_flag'] = (high_limit & low_usage & long_inactive).astype(int)
    
    # 10. v4_point_activity_intensity
    point_cols = ['포인트_적립포인트_R12M', '포인트_이용포인트_R12M']
    points = sum([df_fe[c] if c in df_fe.columns else 0 for c in point_cols])
    usage_base = sum([df_fe[c] if c in df_fe.columns else 0 for c in ['이용금액_일시불_R12M', '이용금액_할부_R12M']])
    df_fe['v4_point_activity_intensity'] = points / (usage_base + 1e-6)
    
    # 11. v4_travel_mileage_activity
    mile_cols = ['마일_적립포인트_R12M', '마일_이용포인트_R12M']
    miles = sum([df_fe[c] if c in df_fe.columns else 0 for c in mile_cols])
    usage_r12_lump = df_fe['이용금액_일시불_R12M'] if '이용금액_일시불_R12M' in df_fe.columns else 1
    df_fe['v4_travel_mileage_activity'] = miles / (usage_r12_lump + 1e-6)
    
    # 12. v4_lifestyle_auto_payment_flag
    telecom = df_fe['납부_통신비이용금액'] if '납부_통신비이용금액' in df_fe.columns else 1
    transport = df_fe['교통_주유이용금액'] if '교통_주유이용금액' in df_fe.columns else 1
    df_fe['v4_lifestyle_auto_payment_flag'] = ((telecom == 0) & (transport == 0)).astype(int)
    
    # 13. v4_arrears_recent_flag
    arrears = df_fe['연체일수_최근'] if '연체일수_최근' in df_fe.columns else 0
    df_fe['v4_arrears_recent_flag'] = (arrears > 30).astype(int)
    
    # 14. v4_cardloan_cleanup_flag
    if all(c in df_fe.columns for c in ['카드론이용금액_누적', '잔액_카드론_B0M', '최종이용일자_카드론']):
        loan_used = df_fe['카드론이용금액_누적'] > 1000000
        loan_cleared = df_fe['잔액_카드론_B0M'] == 0
        loan_long_ago = days_diff(df_fe['최종이용일자_카드론']) > 365
        df_fe['v4_cardloan_cleanup_flag'] = (loan_used & loan_cleared & loan_long_ago).astype(int)
    else:
        df_fe['v4_cardloan_cleanup_flag'] = 0
    
    # 15. v4_online_offline_usage_ratio_R6M
    online = df_fe['이용금액_온라인_R6M'] if '이용금액_온라인_R6M' in df_fe.columns else 0
    offline = df_fe['이용금액_오프라인_R6M'] if '이용금액_오프라인_R6M' in df_fe.columns else 0
    df_fe['v4_online_offline_usage_ratio_R6M'] = online / (online + offline + 1e-6)
    
    # ========== 추가 파생변수 (16-18) ==========
    
    # 16. v4_dormant_score: 종합 휴면 점수 (0-5)
    dormant_features = [
        df_fe['v4_recent_zero_usage_flag'],
        df_fe['v4_long_inactive_high_limit_flag'],
        (df_fe['v4_last_use_gap_card_all'] > 365).astype(int),
        (df_fe['v4_limit_to_usage_ratio_R12M'] < 0.1).astype(int),
        (df_fe['v4_point_activity_intensity'] == 0).astype(int)
    ]
    df_fe['v4_dormant_score'] = sum(dormant_features)
    
    # 17. v4_high_value_dormant: 고가치 휴면 (한도 상위 20% + 저사용)
    limit_threshold = limit.quantile(0.8)
    high_limit_mask = limit > limit_threshold
    low_usage_mask = usage_r12 < 500000
    df_fe['v4_high_value_dormant'] = (high_limit_mask & low_usage_mask).astype(int)
    
    # 18. v4_usage_drop_ratio: 6M vs 12M 사용 감소율
    df_fe['v4_usage_drop_ratio'] = (usage_r12 - usage_r6 * 2) / (usage_r12 + 1e-6)
    
    # NaN/Inf 처리
    v4_features = [c for c in df_fe.columns if c.startswith('v4_')]
    df_fe[v4_features] = df_fe[v4_features].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    print(f"\n[v4 FE 생성 완료] {len(v4_features)}개 파생변수")
    return df_fe

# FE 적용
df_master_v4 = create_v4_features(df_master)
print(f"최종 피처 수: {df_master_v4.shape[1]}")

## 3. Train/Val Split

In [ ]:
TARGET_COL = "Segment"

X_all = df_master_v4.drop(columns=[TARGET_COL])
y_all = df_master_v4[TARGET_COL]

X_train_all, X_val_all, y_train, y_val = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print("Train:", X_train_all.shape, "Val:", X_val_all.shape)
print("\n[Segment 분포]")
print("Train:")
print(y_train.value_counts().sort_index())
print("\nVal:")
print(y_val.value_counts().sort_index())

# Stage 1 Binary 타깃
y_train_binary = (y_train <= 1).astype(int)
y_val_binary = (y_val <= 1).astype(int)

print(f"\n[Stage 1 Binary 타깃]")
print(f"Train Rare: {y_train_binary.sum()}, Others: {(~y_train_binary.astype(bool)).sum()}")
print(f"Val Rare: {y_val_binary.sum()}, Others: {(~y_val_binary.astype(bool)).sum()}")

## 4. Feature Selection

In [ ]:
top150_features = top150_final['feature'].tolist()
v4_features = [c for c in X_train_all.columns if c.startswith('v4_')]

final_features = list(set(top150_features + v4_features))
final_features = [f for f in final_features if f in X_train_all.columns]

print(f"Top150: {len(top150_features)}개")
print(f"v4 FE: {len(v4_features)}개")
print(f"최종 사용 피처: {len(final_features)}개")

X_train = X_train_all[final_features]
X_val = X_val_all[final_features]

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

## 5. Stage 1: 최적화 (SMOTE + 튜닝 + 앙상블)

### 5.1 SMOTE로 Rare 증강

In [ ]:
print("\n" + "="*60)
print("SMOTE 적용: Rare 샘플 증강")
print("="*60)

print(f"\n적용 전 분포:")
print(f"  Rare: {y_train_binary.sum()}")
print(f"  Others: {(~y_train_binary.astype(bool)).sum()}")

# SMOTE 적용 (Rare를 5,000개로 증강)
smote = SMOTE(
    sampling_strategy={1: 5000},
    random_state=42,
    k_neighbors=3
)

X_train_smote, y_train_binary_smote = smote.fit_resample(X_train, y_train_binary)

print(f"\n적용 후 분포:")
print(f"  Rare: {y_train_binary_smote.sum()}")
print(f"  Others: {(~y_train_binary_smote.astype(bool)).sum()}")
print(f"\nX_train shape: {X_train.shape} → {X_train_smote.shape}")

# Class weight 계산
classes_binary = np.unique(y_train_binary_smote)
class_weights_binary = compute_class_weight(
    class_weight="balanced",
    classes=classes_binary,
    y=y_train_binary_smote
)
class_weights_dict_binary = dict(zip(classes_binary, class_weights_binary))
sample_weights_binary_smote = np.array([class_weights_dict_binary[y] for y in y_train_binary_smote])

print("\n[Class Weights (SMOTE 후)]")
for k, v in class_weights_dict_binary.items():
    label = "Rare" if k == 1 else "Others"
    print(f"Class {k} ({label}): {v:.3f}")

### 5.2 하이퍼파라미터 튜닝

In [ ]:
print("\n" + "="*60)
print("하이퍼파라미터 튜닝 (RandomizedSearchCV)")
print("="*60)

param_distributions = {
    'max_depth': [5, 6, 7, 8],
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.03, 0.05, 0.07],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_base = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

random_search = RandomizedSearchCV(
    xgb_base,
    param_distributions,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\n튜닝 시작 (약 5-10분 소요)...")
random_search.fit(
    X_train_smote, y_train_binary_smote,
    sample_weight=sample_weights_binary_smote
)

print(f"\n✅ 최적 파라미터: {random_search.best_params_}")
print(f"✅ 최적 CV Score: {random_search.best_score_:.4f}")

model_xgb_stage1 = random_search.best_estimator_

### 5.3 LightGBM 추가 학습

In [ ]:
print("\n" + "="*60)
print("LightGBM 학습")
print("="*60)

model_lgb_stage1 = LGBMClassifier(
    objective='binary',
    max_depth=7,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model_lgb_stage1.fit(
    X_train_smote, y_train_binary_smote,
    sample_weight=sample_weights_binary_smote,
    eval_set=[(X_val, y_val_binary)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

print("\n✅ LightGBM 학습 완료")

### 5.4 앙상블 + Threshold 최적화

In [ ]:
print("\n" + "="*60)
print("앙상블 예측 + Threshold 최적화")
print("="*60)

# 앙상블 확률 예측
y_prob_xgb_val = model_xgb_stage1.predict_proba(X_val)[:, 1]
y_prob_lgb_val = model_lgb_stage1.predict_proba(X_val)[:, 1]
y_prob_ensemble_val = (y_prob_xgb_val + y_prob_lgb_val) / 2

# Threshold 최적화
thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0.5

print("\nThreshold 탐색 중...")
for thresh in thresholds:
    y_pred_thresh = (y_prob_ensemble_val >= thresh).astype(int)
    f1_rare = f1_score(y_val_binary, y_pred_thresh, pos_label=1)
    f1_macro = f1_score(y_val_binary, y_pred_thresh, average='macro')
    
    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_threshold = thresh
        best_f1_rare = f1_rare

print(f"\n✅ 최적 Threshold: {best_threshold:.2f}")
print(f"✅ Rare F1 Score: {best_f1_rare:.4f}")
print(f"✅ Macro F1 Score: {best_f1:.4f}")

# 최종 예측
y_pred_stage1_train = (((model_xgb_stage1.predict_proba(X_train)[:, 1] + model_lgb_stage1.predict_proba(X_train)[:, 1]) / 2) >= best_threshold).astype(int)
y_pred_stage1_val = (y_prob_ensemble_val >= best_threshold).astype(int)

print("\n[Stage 1 최종 결과 - Validation]")
print(classification_report(y_val_binary, y_pred_stage1_val,
                          target_names=['Others', 'Rare'],
                          digits=4))

### 5.5 Stage 1 결과 시각화

In [ ]:
cm_stage1 = confusion_matrix(y_val_binary, y_pred_stage1_val)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_stage1, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Others', 'Rare'],
           yticklabels=['Others', 'Rare'])
plt.title('Stage 1 (최적화): Rare vs Others - Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

f1_rare_final = f1_score(y_val_binary, y_pred_stage1_val, pos_label=1)
f1_others_final = f1_score(y_val_binary, y_pred_stage1_val, pos_label=0)
f1_macro_s1 = f1_score(y_val_binary, y_pred_stage1_val, average='macro')

print(f"\n{'='*60}")
print(f"STAGE 1 최종 성능 (SMOTE + 튜닝 + 앙상블 + Threshold)")
print(f"{'='*60}")
print(f"Rare F1:      {f1_rare_final:.4f}")
print(f"Others F1:    {f1_others_final:.4f}")
print(f"Macro F1:     {f1_macro_s1:.4f}")
print(f"Threshold:    {best_threshold:.2f}")
print(f"{'='*60}")

## 6. Stage 2-B: 최적화 (SMOTE + 튜닝)

### 6.1 데이터 준비

In [ ]:
# Stage 1 Others 예측 + 실제 2/3/4만 필터링
train_others_mask = (y_pred_stage1_train == 0)
val_others_mask = (y_pred_stage1_val == 0)

train_actual_others_mask = y_train.isin([2, 3, 4])
val_actual_others_mask = y_val.isin([2, 3, 4])

train_final_mask = train_others_mask & train_actual_others_mask
val_final_mask = val_others_mask & val_actual_others_mask

X_train_others = X_train[train_final_mask]
y_train_others = y_train[train_final_mask]

X_val_others = X_val[val_final_mask]
y_val_others = y_val[val_final_mask]

print(f"Stage 2-B 학습 데이터:")
print(f"  Train: {X_train_others.shape[0]} samples")
print(f"  Val: {X_val_others.shape[0]} samples")
print(f"\nTrain 분포 (원본):")
print(y_train_others.value_counts().sort_index())

# 레이블 매핑
label_mapping = {2: 0, 3: 1, 4: 2}
reverse_mapping = {0: 2, 1: 3, 2: 4}

y_train_others_mapped = y_train_others.map(label_mapping)
y_val_others_mapped = y_val_others.map(label_mapping)

print(f"\nTrain 분포 (매핑 후):")
print(y_train_others_mapped.value_counts().sort_index())

### 6.2 SMOTE (Segment 2 증강)

In [ ]:
print("\n" + "="*60)
print("Stage 2-B SMOTE: Segment 2 증강")
print("="*60)

print(f"\n적용 전 분포:")
print(y_train_others_mapped.value_counts().sort_index())

# Segment 2 (매핑 후 0)를 30,000개로 증강
smote_s2 = SMOTE(
    sampling_strategy={0: 30000},
    random_state=42,
    k_neighbors=5
)

X_train_others_smote, y_train_others_mapped_smote = smote_s2.fit_resample(
    X_train_others, y_train_others_mapped
)

print(f"\n적용 후 분포:")
print(pd.Series(y_train_others_mapped_smote).value_counts().sort_index())

# Class weight
classes_others = np.unique(y_train_others_mapped_smote)
class_weights_others = compute_class_weight(
    class_weight="balanced",
    classes=classes_others,
    y=y_train_others_mapped_smote
)
class_weights_dict_others = dict(zip(classes_others, class_weights_others))
sample_weights_others_smote = np.array([class_weights_dict_others[y] for y in y_train_others_mapped_smote])

print("\n[Class Weights]")
for k, v in class_weights_dict_others.items():
    print(f"Class {k} (Seg {reverse_mapping[k]}): {v:.3f}")

### 6.3 하이퍼파라미터 튜닝 + 학습

In [ ]:
print("\n" + "="*60)
print("Stage 2-B 하이퍼파라미터 튜닝")
print("="*60)

param_distributions_s2 = {
    'max_depth': [5, 6, 7],
    'n_estimators': [300, 500],
    'learning_rate': [0.03, 0.05, 0.07],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9]
}

xgb_base_s2 = XGBClassifier(
    objective='multi:softprob',
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

random_search_s2 = RandomizedSearchCV(
    xgb_base_s2,
    param_distributions_s2,
    n_iter=10,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\n튜닝 시작...")
random_search_s2.fit(
    X_train_others_smote, y_train_others_mapped_smote,
    sample_weight=sample_weights_others_smote
)

print(f"\n✅ 최적 파라미터: {random_search_s2.best_params_}")
print(f"✅ 최적 CV Score: {random_search_s2.best_score_:.4f}")

model_stage2 = random_search_s2.best_estimator_

print("\n✅ Stage 2-B 학습 완료")

### 6.4 Stage 2-B 평가

In [ ]:
# 예측 (매핑 레이블)
y_pred_stage2_train_mapped = model_stage2.predict(X_train_others)
y_pred_stage2_val_mapped = model_stage2.predict(X_val_others)

# 원본 레이블로 복원
y_pred_stage2_train = pd.Series(y_pred_stage2_train_mapped).map(reverse_mapping).values
y_pred_stage2_val = pd.Series(y_pred_stage2_val_mapped).map(reverse_mapping).values

print("\n" + "="*60)
print("STAGE 2-B 결과: Segment 2/3/4")
print("="*60)

print("\n[Validation Set]")
print(classification_report(y_val_others, y_pred_stage2_val,
                          target_names=['Seg 2', 'Seg 3', 'Seg 4'],
                          digits=4))

# Confusion Matrix
cm_stage2 = confusion_matrix(y_val_others, y_pred_stage2_val)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_stage2, annot=True, fmt='d', cmap='Greens',
           xticklabels=['Seg 2', 'Seg 3', 'Seg 4'],
           yticklabels=['Seg 2', 'Seg 3', 'Seg 4'])
plt.title('Stage 2-B (최적화): Segment 2/3/4 - Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

f1_macro_stage2 = f1_score(y_val_others, y_pred_stage2_val, average='macro')
print(f"\n[Stage 2-B Macro F1]: {f1_macro_stage2:.4f}")

## 7. 최종 예측 결합

In [ ]:
# 최종 예측 생성
y_pred_final_val = np.zeros_like(y_val, dtype=int)

# Rare로 예측 → 0
rare_mask_val = (y_pred_stage1_val == 1)
y_pred_final_val[rare_mask_val] = 0

# Others로 예측 → Stage 2-B로 재예측
others_mask_val = (y_pred_stage1_val == 0)
X_val_others_for_pred = X_val[others_mask_val]

y_pred_others_mapped = model_stage2.predict(X_val_others_for_pred)
y_pred_others = pd.Series(y_pred_others_mapped).map(reverse_mapping).values

y_pred_final_val[others_mask_val] = y_pred_others

print("\n" + "="*60)
print("최종 2-STAGE 예측 결과")
print("="*60)

print("\n[예측 분포]")
print(pd.Series(y_pred_final_val).value_counts().sort_index())

print("\n[실제 분포]")
print(y_val.value_counts().sort_index())

In [ ]:
# 최종 평가
y_val_grouped = y_val.copy()
y_val_grouped[y_val_grouped == 1] = 0

print("\n[최종 Classification Report]")
print("(Rare 0+1 통합)")
print(classification_report(y_val_grouped, y_pred_final_val,
                          target_names=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'],
                          digits=4))

# Confusion Matrix
cm_final = confusion_matrix(y_val_grouped, y_pred_final_val)
plt.figure(figsize=(8, 7))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='RdYlGn',
           xticklabels=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'],
           yticklabels=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'])
plt.title('Final 2-Stage Model (최적화) - Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

f1_final = f1_score(y_val_grouped, y_pred_final_val, average='macro')

print(f"\n{'='*70}")
print(f"🎯 최종 성능 (SMOTE + 튜닝 + 앙상블 + Threshold 최적화)")
print(f"{'='*70}")
print(f"Stage 1 Rare F1:      {f1_rare_final:.4f}")
print(f"Stage 2-B Macro F1:   {f1_macro_stage2:.4f}")
print(f"Final Macro F1:       {f1_final:.4f}")
print(f"{'='*70}")
print(f"\n✅ 기대 성과:")
print(f"  - Rare F1: 0.3 → {f1_rare_final:.4f} ({(f1_rare_final/0.3 - 1)*100:.1f}% 향상)")
print(f"  - Final Macro F1: 0.53 → {f1_final:.4f} ({(f1_final/0.53 - 1)*100:.1f}% 향상)")

## 8. Feature Importance

In [ ]:
# Stage 1 Feature Importance
importance_s1 = pd.DataFrame({
    'feature': final_features,
    'importance': model_xgb_stage1.feature_importances_
}).sort_values('importance', ascending=False)

print("\n[Stage 1: Top 20 Features for Rare Detection]")
print(importance_s1.head(20))

plt.figure(figsize=(10, 8))
top20 = importance_s1.head(20)
plt.barh(range(len(top20)), top20['importance'])
plt.yticks(range(len(top20)), top20['feature'])
plt.xlabel('Importance')
plt.title('Stage 1: Top 20 Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# v4 FE 피처 랭킹
v4_imp = importance_s1[importance_s1['feature'].str.startswith('v4_')]
print(f"\n[v4 FE Features - Top 10]")
print(v4_imp.head(10))

## 9. 모델 저장

In [ ]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

# Stage 1 앙상블 모델
with open('../models/v4_optimized_stage1_xgb.pkl', 'wb') as f:
    pickle.dump(model_xgb_stage1, f)

with open('../models/v4_optimized_stage1_lgb.pkl', 'wb') as f:
    pickle.dump(model_lgb_stage1, f)

# Stage 2-B 모델
with open('../models/v4_optimized_stage2b.pkl', 'wb') as f:
    pickle.dump(model_stage2, f)

# 설정 정보
config = {
    'best_threshold': float(best_threshold),
    'label_mapping': label_mapping,
    'reverse_mapping': reverse_mapping,
    'final_features': final_features,
    'performance': {
        'rare_f1': float(f1_rare_final),
        'stage2_macro_f1': float(f1_macro_stage2),
        'final_macro_f1': float(f1_final)
    }
}

with open('../models/v4_optimized_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("\n✅ 모든 모델 및 설정 저장 완료!")
print("  - v4_optimized_stage1_xgb.pkl")
print("  - v4_optimized_stage1_lgb.pkl")
print("  - v4_optimized_stage2b.pkl")
print("  - v4_optimized_config.json")

## 10. 최종 요약

### 적용된 최적화 기법

1. **SMOTE**
   - Stage 1: Rare 149 → 5,000
   - Stage 2-B: Seg 2 증강

2. **하이퍼파라미터 튜닝**
   - RandomizedSearchCV
   - 각 Stage별 최적 파라미터 탐색

3. **앙상블**
   - XGBoost + LightGBM
   - 확률 평균으로 안정성↑

4. **Threshold 최적화**
   - Rare 민감도 조정
   - F1 Score 최대화

5. **v4 FE 확장**
   - 15개 → 18개
   - dormant_score, high_value_dormant 등

### 기대 성능

- **Rare F1**: 0.5+ (기존 0.3 대비 67%↑)
- **Final Macro F1**: 0.58+ (기존 0.53 대비 9%↑)

### 실행 시간

- 총 소요: 약 20-30분
  - SMOTE: 2분
  - 튜닝: 10-15분
  - 앙상블 학습: 5분
  - 평가: 3분